# Chess.com Data Exploration

This cleaned notebook assumes `main.py` has already downloaded games and `insertGames.py`
creates both the raw `matches` table and the enriched `matches_enriched` view.

The notebook reads from the database and visualizes. It does not contain a second custom view definition.


## Imports


In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
#from main import main
#main()

#from mainPeers import main
#main()

## Database connection


In [ ]:
PROJECT_DIR = Path("/home/rata/Documents/chess_bot_stack/chess_com")
DB_PATH = PROJECT_DIR / "data" / "chess_analysis.db"

if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

con = duckdb.connect(str(DB_PATH))
print(f"Connected to: {DB_PATH}")


## Refresh and validate enriched view


In [ ]:
from insertGames import CREATE_ENRICHED_VIEW_SQL

con.execute(CREATE_ENRICHED_VIEW_SQL)

tables_df = con.execute("SHOW TABLES").df()
display(tables_df)

required_columns = {
    "owner_username",
    "uuid",
    "time_class",
    "played_as",
    "opening_family",
    "opening_specific",
    "owner_rating",
    "opponent_rating",
    "rating_bucket",
    "outcome",
    "date_played",
}

view_columns = set(con.execute("DESCRIBE matches_enriched").df()["column_name"])
missing_columns = required_columns - view_columns

if missing_columns:
    raise ValueError(f"matches_enriched is missing required columns: {sorted(missing_columns)}")

display(con.execute("DESCRIBE matches_enriched").df())


## Load base data


In [ ]:
matches_df = con.execute("SELECT * FROM matches").df()
enriched_df = con.execute("SELECT * FROM matches_enriched").df()

display(matches_df.head())
display(enriched_df.head())

print(f"matches shape: {matches_df.shape}")
print(f"matches_enriched shape: {enriched_df.shape}")


## Players


In [ ]:
owners_df = con.execute("""
    SELECT DISTINCT owner_username
    FROM matches
    ORDER BY owner_username
""").df()

all_owners = owners_df["owner_username"].tolist()

display(owners_df)
print(all_owners)


In [ ]:
user = 'ratatwIIsk3r'

## Basic quality checks


In [ ]:
duplicates_df = con.execute("""
    SELECT
        owner_username,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT uuid) AS unique_games,
        COUNT(*) - COUNT(DISTINCT uuid) AS duplicate_rows
    FROM matches
    GROUP BY owner_username
    ORDER BY duplicate_rows DESC, total_rows DESC
""").df()

display(duplicates_df)

summary_df = con.execute("""
    SELECT 
        owner_username,
        COUNT(*) AS total_games,
        SUM(
            CASE
                WHEN white_accuracy IS NOT NULL OR black_accuracy IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS analyzed_games
    FROM matches
    GROUP BY owner_username
    ORDER BY total_games DESC
""").df()

display(summary_df)


## Time controls and results


In [ ]:
time_control_df = con.execute("""
    SELECT 
        owner_username,
        time_class,
        COUNT(*) AS games_played
    FROM matches
    GROUP BY owner_username, time_class
    ORDER BY owner_username, games_played DESC
""").df()

display(time_control_df)

results_df = con.execute("""
    SELECT
        owner_username,
        time_class,
        COUNT(*) AS total,
        SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END) AS draws,
        SUM(CASE WHEN outcome = 'loss' THEN 1 ELSE 0 END) AS losses,
        ROUND(100.0 * SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate,
        ROUND(
            100.0 * (
                SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END)
                + 0.5 * SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END)
            ) / COUNT(*),
            1
        ) AS score_rate
    FROM matches_enriched
    GROUP BY owner_username, time_class
    ORDER BY owner_username, total DESC
""").df()

display(results_df)


In [ ]:
mostPlayed = results_df['total'].max()
mostPlayed

## Opening visualization dataset


In [ ]:
openings_df = con.execute("""
    SELECT *
    FROM matches_enriched
    WHERE LOWER(owner_username) = LOWER(?)
""", [user]).df()

openings_df["date_played"] = pd.to_datetime(openings_df["date_played"])

display(openings_df.head())
print(openings_df.info())


## Visualization helper functions


In [ ]:
def get_opening_stats(
    df: pd.DataFrame,
    group_cols: list[str],
    min_games: int = 10,
) -> pd.DataFrame:
    stats = (
        df.groupby(group_cols)
        .agg(
            games=("uuid", "count"),
            avg_rating=("owner_rating", "mean"),
            wins=("outcome", lambda s: (s == "win").sum()),
            draws=("outcome", lambda s: (s == "draw").sum()),
            losses=("outcome", lambda s: (s == "loss").sum()),
        )
        .reset_index()
    )

    stats["avg_rating"] = stats["avg_rating"].round(0).astype("Int64")
    stats["win_rate"] = (100 * stats["wins"] / stats["games"]).round(1)
    stats["score_rate"] = (
        100 * (stats["wins"] + 0.5 * stats["draws"]) / stats["games"]
    ).round(1)

    stats = stats[stats["games"] >= min_games].copy()

    return stats.sort_values(
        group_cols + ["games"],
        ascending=[True] * len(group_cols) + [False],
    )


def filter_user_side(
    df: pd.DataFrame,
    user: str,
    played_as: str | None = None,
    time_class: str | None = None,
) -> pd.DataFrame:
    out = df[df["owner_username"] == user].copy()

    if played_as is not None:
        out = out[out["played_as"] == played_as]

    if time_class is not None:
        out = out[out["time_class"] == time_class]

    return out


## Top opening families by volume


In [ ]:
def plot_top_opening_families(
    openings_df: pd.DataFrame,
    user: str,
    played_as: str,
    time_class: str | None = None,
    top_n: int = 12,
) -> None:
    data = filter_user_side(openings_df, user, played_as, time_class)

    if data.empty:
        print(f"No data for {user}, {played_as}, time_class={time_class}")
        return

    counts = (
        data["opening_family"]
        .value_counts()
        .head(top_n)
        .sort_values()
    )

    ax = counts.plot.barh(figsize=(10, 6))
    ax.set_title(f"{user}: Top {top_n} Opening Families as {played_as}")
    ax.set_xlabel("Games")
    ax.set_ylabel("Opening Family")
    plt.tight_layout()
    plt.show()


#for user in all_owners:

plot_top_opening_families(openings_df, user, "White", top_n=12)
plot_top_opening_families(openings_df, user, "Black", top_n=12)


## Opening family score-rate tables


In [ ]:
family_stats_df = get_opening_stats(
    openings_df,
    group_cols=["owner_username", "played_as", "opening_family"],
    min_games=20,
)

display(family_stats_df)


## Best opening families by score rate


In [ ]:
def plot_opening_family_score_rate(
    family_stats_df: pd.DataFrame,
    user: str,
    played_as: str,
    top_n: int = 12,
    sort_by: str = "score_rate",
) -> None:
    data = family_stats_df[
        (family_stats_df["owner_username"] == user)
        & (family_stats_df["played_as"] == played_as)
    ].copy()

    if data.empty:
        print(f"No family stats for {user} as {played_as}")
        return

    data = data.sort_values(sort_by, ascending=False).head(top_n)
    data = data.sort_values(sort_by)

    ax = data.plot.barh(
        x="opening_family",
        y=sort_by,
        figsize=(10, 6),
        legend=False,
    )

    ax.set_title(f"{user}: Best Opening Families as {played_as} by {sort_by}")
    ax.set_xlabel(f"{sort_by} (%)")
    ax.set_ylabel("Opening Family")

    for i, row in enumerate(data.itertuples()):
        ax.text(
            getattr(row, sort_by) + 0.5,
            i,
            f"{getattr(row, sort_by)}% / {row.games}g",
            va="center",
        )

    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()


# for user in all_owners:

plot_opening_family_score_rate(family_stats_df, user, "White")
plot_opening_family_score_rate(family_stats_df, user, "Black")


## Result distribution by opening family


In [ ]:
def plot_opening_result_distribution(
    family_stats_df: pd.DataFrame,
    user: str,
    played_as: str,
    top_n: int = 10,
) -> None:
    data = family_stats_df[
        (family_stats_df["owner_username"] == user)
        & (family_stats_df["played_as"] == played_as)
    ].copy()

    if data.empty:
        print(f"No result distribution data for {user} as {played_as}")
        return

    data = data.sort_values("games", ascending=False).head(top_n)

    result_pct = data[["opening_family", "wins", "draws", "losses", "games"]].copy()
    result_pct["win_pct"] = 100 * result_pct["wins"] / result_pct["games"]
    result_pct["draw_pct"] = 100 * result_pct["draws"] / result_pct["games"]
    result_pct["loss_pct"] = 100 * result_pct["losses"] / result_pct["games"]

    plot_df = result_pct.set_index("opening_family")[["win_pct", "draw_pct", "loss_pct"]]
    plot_df = plot_df.sort_values("win_pct")

    ax = plot_df.plot.barh(stacked=True, figsize=(11, 7))

    ax.set_title(f"{user}: Result Distribution by Opening Family as {played_as}")
    ax.set_xlabel("Percentage of Games")
    ax.set_ylabel("Opening Family")
    ax.legend(["Wins", "Draws", "Losses"], title="Result", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()


#for user in all_owners:
plot_opening_result_distribution(family_stats_df, user, "White")
plot_opening_result_distribution(family_stats_df, user, "Black")


## Opening family over time


In [ ]:
def plot_opening_family_over_time(
    openings_df: pd.DataFrame,
    user: str,
    played_as: str,
    time_span: str = "M",
    top_n: int = 6,
) -> None:
    data = filter_user_side(openings_df, user, played_as).copy()

    if data.empty:
        print(f"No data for {user} as {played_as}")
        return

    top_openings = (
        data["opening_family"]
        .value_counts()
        .head(top_n)
        .index
        .tolist()
    )

    data = data[data["opening_family"].isin(top_openings)].copy()
    data["period"] = data["date_played"].dt.to_period(time_span).astype(str)

    period_counts = (
        data.groupby(["period", "opening_family"])
        .size()
        .reset_index(name="games")
    )

    pivot_df = (
        period_counts
        .pivot(index="period", columns="opening_family", values="games")
        .fillna(0)
    )

    ax = pivot_df.plot.line(figsize=(13, 7), marker="o", linewidth=2)

    ax.set_title(f"{user}: Top {top_n} Opening Families Over Time as {played_as}")
    ax.set_xlabel(f"Period ({time_span})")
    ax.set_ylabel("Games")
    ax.legend(title="Opening Family", bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


#for user in all_owners:
plot_opening_family_over_time(openings_df, user, "White", time_span="M", top_n=6)
plot_opening_family_over_time(openings_df, user, "Black", time_span="M", top_n=6)


## Opening choice by rating bucket


In [ ]:
rating_family_df = get_opening_stats(
    openings_df,
    group_cols=["owner_username", "played_as", "rating_bucket", "opening_family"],
    min_games=5,
)

rating_family_df["play_share"] = (
    100
    * rating_family_df["games"]
    / rating_family_df.groupby(["owner_username", "played_as", "rating_bucket"])["games"].transform("sum")
).round(1)

display(rating_family_df)


In [ ]:
def plot_opening_choice_by_rating(
    rating_family_df: pd.DataFrame,
    user: str,
    played_as: str,
    top_n: int = 6,
) -> None:
    data = rating_family_df[
        (rating_family_df["owner_username"] == user)
        & (rating_family_df["played_as"] == played_as)
    ].copy()

    if data.empty:
        print(f"No rating bucket data for {user} as {played_as}")
        return

    top_openings = (
        data.groupby("opening_family")["games"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    data = data[data["opening_family"].isin(top_openings)].copy()

    pivot_df = (
        data.pivot_table(
            index="rating_bucket",
            columns="opening_family",
            values="play_share",
            aggfunc="sum",
            fill_value=0,
        )
        .sort_index()
    )

    ax = pivot_df.plot.bar(stacked=True, figsize=(13, 7))

    ax.set_title(f"{user}: Opening Choice by Rating Bucket as {played_as}")
    ax.set_xlabel("Rating Bucket")
    ax.set_ylabel("Play Share (%)")
    ax.legend(title="Opening Family", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.show()


#for user in all_owners:
plot_opening_choice_by_rating(rating_family_df, user, "White", top_n=6)
plot_opening_choice_by_rating(rating_family_df, user, "Black", top_n=6)


## Score-rate heatmap by rating bucket


In [ ]:
def plot_score_rate_heatmap_by_rating(
    rating_family_df: pd.DataFrame,
    user: str,
    played_as: str,
    top_n: int = 8,
) -> None:
    data = rating_family_df[
        (rating_family_df["owner_username"] == user)
        & (rating_family_df["played_as"] == played_as)
    ].copy()

    if data.empty:
        print(f"No heatmap data for {user} as {played_as}")
        return

    top_openings = (
        data.groupby("opening_family")["games"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    data = data[data["opening_family"].isin(top_openings)].copy()

    heatmap_df = data.pivot_table(
        index="opening_family",
        columns="rating_bucket",
        values="score_rate",
        aggfunc="mean",
    )

    if heatmap_df.empty:
        print(f"No heatmap matrix for {user} as {played_as}")
        return

    fig, ax = plt.subplots(figsize=(13, 7))

    image = ax.imshow(heatmap_df, aspect="auto")

    ax.set_title(f"{user}: Score Rate by Rating Bucket and Opening Family as {played_as}")
    ax.set_xlabel("Rating Bucket")
    ax.set_ylabel("Opening Family")

    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, rotation=45)

    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)

    for i in range(len(heatmap_df.index)):
        for j in range(len(heatmap_df.columns)):
            value = heatmap_df.iloc[i, j]
            if pd.notna(value):
                ax.text(j, i, f"{value:.0f}", ha="center", va="center")

    fig.colorbar(image, ax=ax, label="Score Rate (%)")

    plt.tight_layout()
    plt.show()


#for user in all_owners:
plot_score_rate_heatmap_by_rating(rating_family_df, user, "White", top_n=8)
plot_score_rate_heatmap_by_rating(rating_family_df, user, "Black", top_n=8)


## Specific opening drilldown


In [ ]:
specific_stats_df = get_opening_stats(
    openings_df,
    group_cols=["owner_username", "played_as", "opening_family", "opening_specific"],
    min_games=10,
)

display(specific_stats_df)


In [ ]:
def plot_specific_openings_inside_family(
    specific_stats_df: pd.DataFrame,
    user: str,
    played_as: str,
    opening_family: str,
    top_n: int = 12,
    sort_by: str = "games",
) -> None:
    data = specific_stats_df[
        (specific_stats_df["owner_username"] == user)
        & (specific_stats_df["played_as"] == played_as)
        & (specific_stats_df["opening_family"] == opening_family)
    ].copy()

    if data.empty:
        print(f"No specific opening data for {user}, {played_as}, {opening_family}")
        return

    data = data.sort_values(sort_by, ascending=False).head(top_n)
    data = data.sort_values(sort_by)

    ax = data.plot.barh(
        x="opening_specific",
        y=sort_by,
        figsize=(12, 7),
        legend=False,
    )

    ax.set_title(f"{user}: {opening_family} Specific Lines as {played_as}")
    ax.set_xlabel(sort_by)
    ax.set_ylabel("Specific Opening")

    for i, row in enumerate(data.itertuples()):
        label = f"{row.games}g | WR {row.win_rate}% | SR {row.score_rate}%"
        ax.text(getattr(row, sort_by) + 0.5, i, label, va="center")

    plt.tight_layout()
    plt.show()


plot_specific_openings_inside_family(specific_stats_df, 'ratatwIIsk3r', 'White', 'Reti Opening', 12)


## Report table


In [ ]:
def opening_report_table(
    family_stats_df: pd.DataFrame,
    user: str,
    played_as: str,
    min_games: int = 20,
) -> pd.DataFrame:
    data = family_stats_df[
        (family_stats_df["owner_username"] == user)
        & (family_stats_df["played_as"] == played_as)
        & (family_stats_df["games"] >= min_games)
    ].copy()

    cols = [
        "opening_family",
        "games",
        "avg_rating",
        "wins",
        "draws",
        "losses",
        "win_rate",
        "score_rate",
    ]

    return data[cols].sort_values(["score_rate", "games"], ascending=[False, False])


#for user in all_owners:
print(f"\n{user} - White")
display(opening_report_table(family_stats_df, user, "White", min_games=20))

print(f"\n{user} - Black")
display(opening_report_table(family_stats_df, user, "Black", min_games=20))


## Compare User to Peers

In [ ]:
target_user = "ratatwIIsk3r"
cohort_name = "recent_rapid_pm100"
time_class = "rapid"

### Peers dataset

In [ ]:
cohort_quality_df = con.execute("""
    SELECT
        target_username,
        cohort_name,
        COUNT(DISTINCT peer_username) AS peers,
        ROUND(AVG(target_rating), 0) AS avg_target_rating_at_selection,
        ROUND(AVG(peer_rating), 0) AS avg_peer_rating_at_selection,
        ROUND(AVG(rating_diff), 1) AS avg_rating_diff,
        MIN(rating_diff) AS min_rating_diff,
        MAX(rating_diff) AS max_rating_diff,
        MIN(source_game_date) AS oldest_source_game,
        MAX(source_game_date) AS newest_source_game
    FROM peer_cohort_members
    WHERE LOWER(target_username) = LOWER(?)
      AND cohort_name = ?
    GROUP BY target_username, cohort_name
""", [target_user, cohort_name]).df()

display(cohort_quality_df)

### Peers comparison W/D/L

In [ ]:
comparison_query = """
WITH peers AS (
    SELECT DISTINCT peer_username
    FROM peer_cohort_members
    WHERE LOWER(target_username) = LOWER(?)
      AND cohort_name = ?
),
base AS (
    SELECT
        CASE
            WHEN LOWER(owner_username) = LOWER(?) THEN 'target'
            WHEN owner_username IN (SELECT peer_username FROM peers) THEN 'peers'
            ELSE NULL
        END AS group_name,
        *
    FROM matches_enriched
    WHERE time_class = ?
      AND (
            LOWER(owner_username) = LOWER(?)
            OR owner_username IN (SELECT peer_username FROM peers)
      )
)
SELECT
    group_name,
    COUNT(*) AS games,
    ROUND(AVG(owner_rating), 0) AS avg_rating,

    SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END) AS draws,
    SUM(CASE WHEN outcome = 'loss' THEN 1 ELSE 0 END) AS losses,

    ROUND(
        100.0 * SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END) / COUNT(*),
        1
    ) AS win_rate,

    ROUND(
        100.0 * (
            SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END)
            + 0.5 * SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END)
        ) / COUNT(*),
        1
    ) AS score_rate

FROM base
WHERE group_name IS NOT NULL
GROUP BY group_name
ORDER BY group_name
"""

comparison_df = con.execute(
    comparison_query,
    [target_user, cohort_name, target_user, time_class, target_user],
).df()

display(comparison_df)

### Peers Openings

In [ ]:
opening_peer_query = """
WITH peers AS (
    SELECT DISTINCT peer_username
    FROM peer_cohort_members
    WHERE LOWER(target_username) = LOWER(?)
      AND cohort_name = ?
),
base AS (
    SELECT
        CASE
            WHEN LOWER(owner_username) = LOWER(?) THEN 'target'
            WHEN owner_username IN (SELECT peer_username FROM peers) THEN 'peers'
            ELSE NULL
        END AS group_name,
        played_as,
        opening_family,
        outcome,
        uuid
    FROM matches_enriched
    WHERE time_class = ?
      AND opening_family IS NOT NULL
      AND opening_family != ''
      AND (
            LOWER(owner_username) = LOWER(?)
            OR owner_username IN (SELECT peer_username FROM peers)
      )
),
grouped AS (
    SELECT
        group_name,
        played_as,
        opening_family,
        COUNT(*) AS games,
        SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END) AS draws,
        SUM(CASE WHEN outcome = 'loss' THEN 1 ELSE 0 END) AS losses
    FROM base
    WHERE group_name IS NOT NULL
    GROUP BY group_name, played_as, opening_family
)
SELECT
    *,
    ROUND(100.0 * games / SUM(games) OVER (PARTITION BY group_name, played_as), 1) AS play_share,
    ROUND(100.0 * wins / games, 1) AS win_rate,
    ROUND(100.0 * (wins + 0.5 * draws) / games, 1) AS score_rate
FROM grouped
WHERE games >= ?
ORDER BY group_name, played_as, games DESC
"""

opening_peer_df = con.execute(
    opening_peer_query,
    [target_user, cohort_name, target_user, time_class, target_user, 10],
).df()

display(opening_peer_df)

In [ ]:
comparison_openings_df = con.execute("""
    WITH peers AS (
        SELECT DISTINCT LOWER(peer_username) AS peer_username
        FROM peer_cohort_members
        WHERE LOWER(target_username) = LOWER(?)
          AND cohort_name = ?
    )
    SELECT
        CASE
            WHEN LOWER(owner_username) = LOWER(?) THEN 'target'
            ELSE 'peers'
        END AS comparison_group,

        owner_username,
        uuid,
        time_class,
        played_as,
        opening_family,
        opening_specific,
        owner_rating,
        opponent_username,
        opponent_rating,
        rating_bucket,
        outcome,
        date_played

    FROM matches_enriched
    WHERE time_class = ?
      AND opening_family IS NOT NULL
      AND opening_family != ''
      AND (
            LOWER(owner_username) = LOWER(?)
            OR LOWER(owner_username) IN (SELECT peer_username FROM peers)
      )
""", [target_user, cohort_name, target_user, time_class, target_user]).df()

comparison_openings_df["date_played"] = pd.to_datetime(comparison_openings_df["date_played"])

display(comparison_openings_df.head())
display(comparison_openings_df["comparison_group"].value_counts())
display(comparison_openings_df.shape)

In [ ]:
overall_comparison_df = (
    comparison_openings_df
    .groupby("comparison_group")
    .agg(
        games=("uuid", "count"),
        avg_rating=("owner_rating", "mean"),
        wins=("outcome", lambda s: (s == "win").sum()),
        draws=("outcome", lambda s: (s == "draw").sum()),
        losses=("outcome", lambda s: (s == "loss").sum()),
    )
    .reset_index()
)

overall_comparison_df["avg_rating"] = overall_comparison_df["avg_rating"].round(0)
overall_comparison_df["win_rate"] = (
    100 * overall_comparison_df["wins"] / overall_comparison_df["games"]
).round(1)

overall_comparison_df["draw_rate"] = (
    100 * overall_comparison_df["draws"] / overall_comparison_df["games"]
).round(1)

overall_comparison_df["loss_rate"] = (
    100 * overall_comparison_df["losses"] / overall_comparison_df["games"]
).round(1)

overall_comparison_df["score_rate"] = (
    100 * (
        overall_comparison_df["wins"] + 0.5 * overall_comparison_df["draws"]
    ) / overall_comparison_df["games"]
).round(1)

display(overall_comparison_df)

In [ ]:
opening_comparison_df = (
    comparison_openings_df
    .groupby(["comparison_group", "played_as", "opening_family"])
    .agg(
        games=("uuid", "count"),
        wins=("outcome", lambda s: (s == "win").sum()),
        draws=("outcome", lambda s: (s == "draw").sum()),
        losses=("outcome", lambda s: (s == "loss").sum()),
    )
    .reset_index()
)

opening_comparison_df["play_share"] = (
    100
    * opening_comparison_df["games"]
    / opening_comparison_df.groupby(["comparison_group", "played_as"])["games"].transform("sum")
).round(1)

opening_comparison_df["win_rate"] = (
    100 * opening_comparison_df["wins"] / opening_comparison_df["games"]
).round(1)

opening_comparison_df["score_rate"] = (
    100
    * (opening_comparison_df["wins"] + 0.5 * opening_comparison_df["draws"])
    / opening_comparison_df["games"]
).round(1)

display(
    opening_comparison_df
    .sort_values(["comparison_group", "played_as", "games"], ascending=[True, True, False])
)

In [ ]:
def plot_target_vs_peers_opening_share(
    opening_comparison_df: pd.DataFrame,
    played_as: str,
    top_n: int = 10,
) -> None:
    data = opening_comparison_df[
        opening_comparison_df["played_as"] == played_as
    ].copy()

    if data.empty:
        print(f"No opening comparison data for {played_as}")
        return

    top_openings = (
        data[data["comparison_group"] == "target"]
        .sort_values("games", ascending=False)
        .head(top_n)["opening_family"]
        .tolist()
    )

    if not top_openings:
        print(f"No target openings found for {played_as}")
        return

    data = data[data["opening_family"].isin(top_openings)]

    pivot_df = data.pivot_table(
        index="opening_family",
        columns="comparison_group",
        values="play_share",
        fill_value=0,
    )

    if "target" in pivot_df.columns:
        pivot_df = pivot_df.sort_values("target", ascending=True)

    ax = pivot_df.plot.barh(figsize=(11, 7))
    ax.set_title(f"Opening Family Play Share: Target vs Peers as {played_as}")
    ax.set_xlabel("Play Share (%)")
    ax.set_ylabel("Opening Family")
    ax.legend(title="Group")

    plt.tight_layout()
    plt.show()

plot_target_vs_peers_opening_share(opening_comparison_df, "White", top_n=10)
plot_target_vs_peers_opening_share(opening_comparison_df, "Black", top_n=10)

In [ ]:
opening_gap_df = opening_comparison_df.pivot_table(
    index=["played_as", "opening_family"],
    columns="comparison_group",
    values="play_share",
    fill_value=0,
).reset_index()

opening_gap_df.columns.name = None

if "target" not in opening_gap_df.columns:
    opening_gap_df["target"] = 0

if "peers" not in opening_gap_df.columns:
    opening_gap_df["peers"] = 0

opening_gap_df["target_minus_peers"] = (
    opening_gap_df["target"] - opening_gap_df["peers"]
).round(1)

display(
    opening_gap_df
    .sort_values("target_minus_peers", ascending=False)
    .head(20)
)

In [ ]:
def plot_opening_overuse(opening_gap_df: pd.DataFrame, played_as: str, top_n: int = 10) -> None:
    data = opening_gap_df[opening_gap_df["played_as"] == played_as].copy()
    data = data.sort_values("target_minus_peers", ascending=False).head(top_n)
    data = data.sort_values("target_minus_peers")

    ax = data.plot.barh(
        x="opening_family",
        y="target_minus_peers",
        figsize=(10, 6),
        legend=False,
    )

    ax.set_title(f"{target_user}: Most Overplayed Opening Families vs Peers as {played_as}")
    ax.set_xlabel("Target play share minus peer play share (%)")
    ax.set_ylabel("Opening Family")
    plt.tight_layout()
    plt.show()


plot_opening_overuse(opening_gap_df, "White")
plot_opening_overuse(opening_gap_df, "Black")

In [ ]:
def plot_opening_underuse(opening_gap_df: pd.DataFrame, played_as: str, top_n: int = 10) -> None:
    data = opening_gap_df[opening_gap_df["played_as"] == played_as].copy()
    data = data.sort_values("target_minus_peers", ascending=True).head(top_n)
    data = data.sort_values("target_minus_peers")

    ax = data.plot.barh(
        x="opening_family",
        y="target_minus_peers",
        figsize=(10, 6),
        legend=False,
    )

    ax.set_title(f"{target_user}: Most Underplayed Opening Families vs Peers as {played_as}")
    ax.set_xlabel("Target play share minus peer play share (%)")
    ax.set_ylabel("Opening Family")
    plt.tight_layout()
    plt.show()


plot_opening_underuse(opening_gap_df, "White")
plot_opening_underuse(opening_gap_df, "Black")

In [ ]:
opening_score_gap_df = opening_comparison_df.pivot_table(
    index=["played_as", "opening_family"],
    columns="comparison_group",
    values="score_rate",
).reset_index()

opening_score_gap_df.columns.name = None

opening_games_df = opening_comparison_df.pivot_table(
    index=["played_as", "opening_family"],
    columns="comparison_group",
    values="games",
    fill_value=0,
).reset_index()

opening_games_df.columns.name = None

opening_score_gap_df = opening_score_gap_df.merge(
    opening_games_df,
    on=["played_as", "opening_family"],
    suffixes=("_score", "_games"),
)

required_cols = ["target_score", "peers_score", "target_games", "peers_games"]
for col in required_cols:
    if col not in opening_score_gap_df.columns:
        opening_score_gap_df[col] = pd.NA

opening_score_gap_df["score_gap"] = (
    opening_score_gap_df["target_score"] - opening_score_gap_df["peers_score"]
).round(1)

display(
    opening_score_gap_df
    .dropna(subset=["target_score", "peers_score"])
    .sort_values("score_gap", ascending=False)
)

In [ ]:
def plot_opening_score_gap(score_gap_df: pd.DataFrame, played_as: str, top_n: int = 10) -> None:
    data = score_gap_df[
        (score_gap_df["played_as"] == played_as)
        & (score_gap_df["target_games"] >= 10)
        & (score_gap_df["peers_games"] >= 20)
    ].copy()

    data = data.sort_values("score_gap", ascending=False).head(top_n)
    data = data.sort_values("score_gap")

    ax = data.plot.barh(
        x="opening_family",
        y="score_gap",
        figsize=(10, 6),
        legend=False,
    )

    ax.set_title(f"{target_user}: Best Opening Score Gaps vs Peers as {played_as}")
    ax.set_xlabel("Target score rate minus peer score rate (%)")
    ax.set_ylabel("Opening Family")
    plt.tight_layout()
    plt.show()


plot_opening_score_gap(opening_score_gap_df, "White")
plot_opening_score_gap(opening_score_gap_df, "Black")

In [ ]:
repertoire_width_df = (
    comparison_openings_df
    .groupby(["comparison_group", "played_as"])
    .agg(
        games=("uuid", "count"),
        unique_families=("opening_family", "nunique"),
        unique_specific_openings=("opening_specific", "nunique"),
    )
    .reset_index()
)

repertoire_width_df["games_per_family"] = (
    repertoire_width_df["games"] / repertoire_width_df["unique_families"]
).round(1)

display(repertoire_width_df)

In [ ]:
ax = repertoire_width_df.pivot(
    index="played_as",
    columns="comparison_group",
    values="unique_families",
).plot.bar(figsize=(8, 5))

ax.set_title(f"{target_user}: Repertoire Width vs Peers")
ax.set_ylabel("Unique Opening Families")
ax.set_xlabel("Played As")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
def calculate_top_n_opening_share(df: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    grouped = (
        df.groupby(["comparison_group", "played_as", "opening_family"])
        .size()
        .reset_index(name="games")
    )

    total = (
        grouped.groupby(["comparison_group", "played_as"])["games"]
        .sum()
        .reset_index(name="total_games")
    )

    top_n = (
        grouped.sort_values(["comparison_group", "played_as", "games"], ascending=[True, True, False])
        .groupby(["comparison_group", "played_as"])
        .head(n)
        .groupby(["comparison_group", "played_as"])["games"]
        .sum()
        .reset_index(name=f"top_{n}_games")
    )

    result = total.merge(top_n, on=["comparison_group", "played_as"])
    result[f"top_{n}_share"] = (
        100 * result[f"top_{n}_games"] / result["total_games"]
    ).round(1)

    return result


top3_share_df = calculate_top_n_opening_share(comparison_openings_df, n=3)
display(top3_share_df)

In [ ]:
result_reason_df = con.execute("""
    WITH peers AS (
        SELECT DISTINCT LOWER(peer_username) AS peer_username
        FROM peer_cohort_members
        WHERE LOWER(target_username) = LOWER(?)
          AND cohort_name = ?
    ),
    base AS (
        SELECT
            CASE
                WHEN LOWER(owner_username) = LOWER(?) THEN 'target'
                ELSE 'peers'
            END AS comparison_group,

            CASE
                WHEN outcome = 'win' AND played_as = 'White' THEN black_result
                WHEN outcome = 'win' AND played_as = 'Black' THEN white_result
                WHEN outcome = 'loss' AND played_as = 'White' THEN white_result
                WHEN outcome = 'loss' AND played_as = 'Black' THEN black_result
                ELSE outcome
            END AS result_reason,

            outcome,
            uuid

        FROM matches_enriched
        WHERE time_class = ?
          AND (
                LOWER(owner_username) = LOWER(?)
                OR LOWER(owner_username) IN (SELECT peer_username FROM peers)
          )
    )
    SELECT
        comparison_group,
        outcome,
        result_reason,
        COUNT(*) AS games,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY comparison_group, outcome),
            1
        ) AS share
    FROM base
    WHERE result_reason IS NOT NULL
    GROUP BY comparison_group, outcome, result_reason
    ORDER BY comparison_group, outcome, games DESC
""", [target_user, cohort_name, target_user, time_class, target_user]).df()

display(result_reason_df)

In [ ]:
rating_bucket_comparison_df = (
    comparison_openings_df
    .groupby(["comparison_group", "rating_bucket"])
    .agg(
        games=("uuid", "count"),
        avg_rating=("owner_rating", "mean"),
        wins=("outcome", lambda s: (s == "win").sum()),
        draws=("outcome", lambda s: (s == "draw").sum()),
        losses=("outcome", lambda s: (s == "loss").sum()),
    )
    .reset_index()
)

rating_bucket_comparison_df["score_rate"] = (
    100
    * (rating_bucket_comparison_df["wins"] + 0.5 * rating_bucket_comparison_df["draws"])
    / rating_bucket_comparison_df["games"]
).round(1)

display(rating_bucket_comparison_df)

In [ ]:
pivot_df = rating_bucket_comparison_df.pivot(
    index="rating_bucket",
    columns="comparison_group",
    values="score_rate",
)

ax = pivot_df.plot.line(figsize=(10, 6), marker="o")
ax.set_title(f"{target_user}: Score Rate by Rating Bucket vs Peers")
ax.set_xlabel("Rating Bucket")
ax.set_ylabel("Score Rate (%)")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
time_class_profile_df = con.execute("""
    WITH peers AS (
        SELECT DISTINCT LOWER(peer_username) AS peer_username
        FROM peer_cohort_members
        WHERE LOWER(target_username) = LOWER(?)
          AND cohort_name = ?
    ),
    base AS (
        SELECT
            CASE
                WHEN LOWER(owner_username) = LOWER(?) THEN 'target'
                ELSE 'peers'
            END AS comparison_group,
            time_class,
            uuid
        FROM matches_enriched
        WHERE LOWER(owner_username) = LOWER(?)
           OR LOWER(owner_username) IN (SELECT peer_username FROM peers)
    )
    SELECT
        comparison_group,
        time_class,
        COUNT(*) AS games,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY comparison_group),
            1
        ) AS share
    FROM base
    GROUP BY comparison_group, time_class
    ORDER BY comparison_group, games DESC
""", [target_user, cohort_name, target_user, target_user]).df()

display(time_class_profile_df)

In [ ]:
pivot_df = time_class_profile_df.pivot(
    index="time_class",
    columns="comparison_group",
    values="share",
).fillna(0)

ax = pivot_df.plot.bar(figsize=(9, 5))
ax.set_title(f"{target_user}: Time-Control Profile vs Peers")
ax.set_ylabel("Share of Games (%)")
ax.set_xlabel("Time Class")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
peer_leaderboard_df = con.execute("""
    WITH cohort AS (
        SELECT DISTINCT LOWER(peer_username) AS peer_username
        FROM peer_cohort_members
        WHERE LOWER(target_username) = LOWER(?)
          AND cohort_name = ?
    )
    SELECT
        owner_username,
        COUNT(*) AS games,
        ROUND(AVG(owner_rating), 0) AS avg_rating,
        SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END) AS draws,
        SUM(CASE WHEN outcome = 'loss' THEN 1 ELSE 0 END) AS losses,
        ROUND(
            100.0 * (
                SUM(CASE WHEN outcome = 'win' THEN 1 ELSE 0 END)
                + 0.5 * SUM(CASE WHEN outcome = 'draw' THEN 1 ELSE 0 END)
            ) / COUNT(*),
            1
        ) AS score_rate
    FROM matches_enriched
    WHERE time_class = ?
      AND LOWER(owner_username) IN (SELECT peer_username FROM cohort)
    GROUP BY owner_username
    ORDER BY score_rate DESC
""", [target_user, cohort_name, time_class]).df()

display(peer_leaderboard_df)

In [ ]:
target_share = opening_gap_df[
    opening_gap_df["played_as"] == "White"
][["opening_family", "target", "peers", "target_minus_peers"]].copy()

target_share["abs_gap"] = target_share["target_minus_peers"].abs()

display(target_share.sort_values("abs_gap", ascending=False).head(15))

## Close connection


In [ ]:
con.close()
